# Credit Risk Default Prediction
### LendingClub Loan Data · 2007–2018 · 2.26M Loans

---

## Problem

When a bank or lending platform issues a loan, it faces a fundamental
uncertainty: will the borrower repay? A missed default costs the lender
the remaining principal. A wrongly rejected good borrower costs revenue
and damages customer relationships.

This project builds a machine learning system to predict loan default
**at the time of origination** — using only information available before
the loan is issued — to help lenders make better approval decisions.

---

## Dataset

**Source**: LendingClub loan data (Kaggle), 2007–2018  
**Size**: 2.26M loans · 151 raw features  
**Target**: Binary — `1` = Default/Charged Off, `0` = Fully Paid  
**Base default rate**: ~22%

---

## Approach

| Stage | Key Decisions |
|-------|--------------|
| **Data cleaning** | Removed 35+ columns (post-outcome leakage, low-variance, high-missing); handled missing data via block pattern analysis |
| **Feature engineering** | 15+ financial ratios, credit behavior flags, leakage-safe target encoding |
| **Modeling** | XGBoost with `scale_pos_weight` for class imbalance |
| **Evaluation** | ROC-AUC, Average Precision, KS Statistic, threshold analysis |
| **Explainability** | SHAP feature importance + single-prediction explanation |

---

## Results

| Metric | Score |
|--------|-------|
| **ROC-AUC** | 0.74 |
| **Average Precision** | 0.439 |
| **KS Statistic** | 0.349 |
| **Recall @ threshold 0.530** | 64% of defaults caught |
| **Good borrowers approved** | 71% |

> At the optimal threshold of 0.530, the model catches 64% of defaults
> while approving 71% of creditworthy borrowers — outperforming a random
> baseline by ~15 percentage points in precision.

---

## Structure

1. Data loading & memory optimization  
2. Leakage removal & feature selection  
3. Target definition & class imbalance  
4. Missing data analysis  
5. Feature engineering  
6. Model training & pipeline  
7. Evaluation & threshold analysis  
8. SHAP explainability

In [ ]:
# Cell 1: Install kaggle
!pip install kaggle -q

### API json file creation

In [ ]:
import json
import os

kaggle_credentials = {
    "username": "codehashira",   # replace this
    "key": "KGAT_4e19b730d1d15de8ee5f72a22d7c5ec8"            # replace this
}


os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

kaggle_path = os.path.expanduser('~/.kaggle/kaggle.json')

with open(kaggle_path, 'w') as f:
    json.dump(kaggle_credentials, f)

os.chmod(kaggle_path, 0o600)

print("kaggle.json created successfully")

kaggle.json created successfully


### Downloading and loading the dataset

In [ ]:
# Cell 4: Download Lending Club dataset
!kaggle datasets download \
    -d wordsforthewise/lending-club

Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
License(s): CC0-1.0
 60% 771M/1.26G [00:19<00:13, 41.0MB/s]

In [ ]:
# Cell 2: Check that the zip file exists
!ls -lh lending-club.zip

# Cell 3: Unzip
!mkdir -p ./data
!unzip -o lending-club.zip -d ./data/

# Cell 4: See what's inside
!ls -lh ./data/

### Loading dataset

In [ ]:
import pandas as pd
df = pd.read_csv(
    './data/accepted_2007_to_2018Q4.csv.gz',
    low_memory=False
)
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")

In [ ]:
def reduce_memory(df):
    """Shrink dataframe memory by downcasting types"""

    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        if col_type == 'float64':
            df[col] = df[col].astype('float32')

        elif col_type == 'int64':
            col_min = df[col].min()
            col_max = df[col].max()

            if col_min >= 0:
                # Unsigned integers (no negatives)
                if col_max <= 255:
                    df[col] = df[col].astype('uint8')
                elif col_max <= 65535:
                    df[col] = df[col].astype('uint16')
                elif col_max <= 4294967295:
                    df[col] = df[col].astype('uint32')
            else:
                # Signed integers (has negatives)
                if col_min >= -128 and col_max <= 127:
                    df[col] = df[col].astype('int8')
                elif col_min >= -32768 and col_max <= 32767:
                    df[col] = df[col].astype('int16')
                elif col_min >= -2147483648 and col_max <= 2147483647:
                    df[col] = df[col].astype('int32')

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    reduction = (start_mem - end_mem) / start_mem * 100

    print(f"Memory: {start_mem:.0f} MB → {end_mem:.0f} MB "
          f"({reduction:.0f}% reduction)")

    return df

# Reduce memory immediately
df = reduce_memory(df)

In [ ]:
print("DATA TYPES AFTER REDUCTION:")
print(df.dtypes.value_counts())
print(f"\nTotal memory: "
      f"{df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

In [ ]:
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'percentage': df.isnull().mean()
})

missing = missing[missing['count'] > 0]
missing = missing.sort_values('percentage', ascending=False)

print(f"Columns with missing values: "
      f"{len(missing)} out of {df.shape[1]}")
print("=" * 60)

for col, row in missing.iterrows():
    bar = "█" * int(row['percentage'] * 50)
    print(f"{col:60s} {row['percentage']:6.1%}  {bar}")

### Check columns for slightly empty and investigate its values

In [ ]:
slightly_empty = missing[
    missing['percentage'] < 0.85
].index.tolist()

print(f"SLIGHTLY EMPTY ({len(slightly_empty)} columns):")
print("=" * 50)

for col in slightly_empty:
    pct = df[col].isnull().mean()
    print(f"  {col:40s} {pct:.1%}")

for col in slightly_empty:
    print(f"\n{col}:")
    print(f"  Missing: {df[col].isnull().mean():.1%}")
    print(f"  Non-null values sample:")
    sample = df[col].dropna().head(5)
    if len(sample) > 0:
        print(f"  {sample.values}")
df.shape

### Leakage and Useless Columns Removed
These 19 columns contain information only available
AFTER a loan outcome is determined. Including them
would create data leakage — the model would learn
to detect defaults by seeing that recovery fees
were charged (which only happens AFTER a default).
In production, none of this data would be available
at the time of the lending decision.

In [ ]:
leakage_cols = [
    # Payment information - only known after
    # the loan is being repaid or has ended
    'total_pymnt',             # total amount paid
    'total_pymnt_inv',         # total paid by investors
    'total_rec_prncp',         # principal received
    'total_rec_int',           # interest received
    'total_rec_late_fee',      # late fees received
    'last_pymnt_d',            # date of last payment
    'last_pymnt_amnt',         # amount of last payment
    'next_pymnt_d',            # next payment date

    # Recovery - only exists for defaulted loans
    'recoveries',              # money recovered after default
    'collection_recovery_fee', # fees from recovery

    # Outstanding balance - only known during repayment
    'out_prncp',               # remaining principal
    'out_prncp_inv',           # remaining principal for investors

    # Post-origination credit checks
    'last_credit_pull_d',      # when credit was last pulled
    'last_fico_range_high',    # most recent FICO score
    'last_fico_range_low',     # most recent FICO score

    # Post-origination flags
    'hardship_flag',           # entered hardship program
    'debt_settlement_flag',    # entered debt settlement
    'settlement_date',
    'settlement_status',
    'debt_settlement_flag_date',
]
useless_cols = [
  'id',                  # Just the ids
  'member_id',
  'url',                # just a link to the listing
  'title',              # free text loan title, messy duplicate of 'purpose'
  'emp_title',          # 500,000+ unique job titles, too messy to use directly
  'pymnt_plan',         # almost all "n", no variance
  'policy_code',        # all 1s, no variance
  'zip_code',           # only first 3 digits, use addr_state instead
  'funded_amnt',        # almost identical to loan_amnt
  'funded_amnt_inv',    # almost identical to loan_amnt
  'disbursement_method' # The percentage of Directpay that defaulted is incredibly small at 0.2%
]
cols_to_drop = leakage_cols + useless_cols
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
credit_detail_cols = [
    'il_util',                # installment loan utilization
    'mths_since_rcnt_il',     # months since recent installment loan
    'all_util',               # utilization on all accounts
    'open_il_24m',            # installment loans opened in 24 months
    'total_cu_tl',            # total credit union tradelines
    'total_bal_il',           # total installment loan balance
    'inq_fi',                 # inquiries at finance companies
    'open_acc_6m',            # accounts opened in 6 months
    'open_act_il',            # active installment loans
    'max_bal_bc',             # max balance on bank cards
    'open_rv_24m',            # revolving accounts opened in 24 months
    'inq_last_12m',           # inquiries in last 12 months
    'open_rv_12m',            # revolving accounts opened in 12 months
    'open_il_12m',            # installment loans opened in 12 months
]

df.drop(columns=credit_detail_cols,
        inplace=True, errors='ignore')

## Class Imbalance

The dataset has a **22% default rate** — meaning roughly 1 in 5 loans
ended in default. This imbalance drives every downstream modeling decision:

- **Metric selection**: Accuracy is misleading here — a model that predicts
  "paid" for every loan achieves 78% accuracy while catching zero defaults.
  ROC-AUC and Average Precision are used instead.

- **Class weighting**: `scale_pos_weight` is set to the ratio of negative
  to positive samples (~3.6x), telling XGBoost to penalize missed defaults
  more heavily during training.

- **Stratified splits**: `train_test_split(stratify=y)` ensures the 22%
  default rate is preserved identically in both train and test sets.

- **Threshold selection**: The default 0.5 threshold is suboptimal on
  imbalanced data. A threshold of 0.530 was selected by maximizing F1
  score on the test set.

In [ ]:
# Default = loss for the lender
default_statuses = [
    'Charged Off',
    'Default',
    'Late (31-120 days)',
]

# Paid = borrower repaid successfully
paid_statuses = [
    'Fully Paid'
]

# Drop ambiguous statuses
df = df[
    df['loan_status'].isin(
        default_statuses + paid_statuses
    )
].copy()

# Create binary target
df['default'] = df['loan_status'].isin(
    default_statuses
).astype(int)

# Verify
df.drop(columns='loan_status', inplace=True, errors ='ignore')
print(f"Rows remaining: {len(df):,}")
print(f"Rows dropped: {2260701 - len(df):,}")
print(f"\nTarget distribution:")
print(df['default'].value_counts())
print(f"\nDefault rate: {df['default'].mean():.2%}")

### Transform Dates

In [ ]:
df['earliest_cr_line'] = pd.to_datetime(
    df['earliest_cr_line'], format='%b-%Y'
)

# issue_d — 15 unique dates (month-year of loan issue)
df['issue_d'] = pd.to_datetime(
    df['issue_d'], format='%b-%Y'
)

# Create useful features from both
df['credit_history_years'] = (
    (df['issue_d'] - df['earliest_cr_line']).dt.days / 365
).round(1)

df['issue_year'] = df['issue_d'].dt.year
df['issue_month'] = df['issue_d'].dt.month

# Drop the original date columns
df = df.drop(columns=['earliest_cr_line', 'issue_d'])

print("Date columns transformed")
print(f"  credit_history_years: "
      f"{df['credit_history_years'].mean():.1f} "
      f"years average")

# Investigate deliquency columns

In [ ]:
delinquency_cols = [
    'mths_since_last_delinq',          # any missed payment
    'mths_since_last_record',          # bankruptcy, tax lien
    'mths_since_last_major_derog',     # charged off, collections
    'mths_since_recent_bc_dlq',        # missed credit card payment
    'mths_since_recent_revol_delinq',  # missed revolving payment
    'mths_since_recent_inq',           # credit was checked
]
df[f'has_deliq'] = df[delinquency_cols].notna().any(axis=1).astype(int)
for col in delinquency_cols:
    if col in df.columns:
        missing_pct = df[col].isnull().mean()
        missing_default = df[df[col].isnull()]['default'].mean()
        present_default = df[df[col].notna()]['default'].mean()

        print(f"{col}:")
        print(f"  Missing: {missing_pct:.1%}")
        print(f"  Default when MISSING (never happened):  "
              f"{missing_default:.2%}")
        print(f"  Default when PRESENT (it happened):     "
              f"{present_default:.2%}")

        # Create flag: has this person ever had this issue?

        # Fill NaN with 999 (never happened = long time ago)
        df[col] = df[col].fillna(999)
print("Done. Flags created, NaN filled with 999.")

### Delinquency Analysis Findings

The strongest signal comes from `mths_since_recent_inq`:
borrowers with no recent credit inquiries default at
15.73% vs 22.03% for those with inquiries — a 6.3
percentage point gap. This suggests that credit-seeking
behavior is a meaningful risk indicator.

`mths_since_last_record` (bankruptcies, tax liens) shows
a 3.44 percentage point gap, confirming that serious
credit events have lasting predictive value.

The remaining delinquency columns show smaller but
consistent differences (1-2 percentage points). While
individually weak, these features may contribute
meaningful signal when combined in a model.

For all six columns, NaN indicates the event never
occurred — the cleanest possible history. A binary flag
were created to capture this, and NaN values were
replaced with 999 to represent "never happened."

### Handling missing data

In [ ]:
missing = df.isnull().mean()
missing = missing[missing > 0].sort_values(ascending=False)

tier1 = missing[missing <= 0.05]

for col, pct in tier1.items():
    print(f"  {col:40s} {pct:.1%}")

# Check: are the same rows missing across all 4.9% columns?
cols_49 = [col for col in df.columns
           if abs(df[col].isnull().mean() - 0.049) < 0.005]

print(f"Columns with ~4.9% missing: {len(cols_49)}")

# Check if the SAME rows are missing
missing_mask = df[cols_49].isnull()
all_missing = missing_mask.all(axis=1)  # missing in ALL columns
any_missing = missing_mask.any(axis=1)  # missing in ANY column

print(f"\nRows missing ALL {len(cols_49)} columns: "
      f"{all_missing.sum():,}")
print(f"Rows missing ANY of them:              "
      f"{any_missing.sum():,}")
print(f"Rows missing SOME but not all:         "
      f"{(any_missing & ~all_missing).sum():,}")
# How many columns are missing for each row?
missing_counts = df[cols_49].isnull().sum(axis=1)

print("MISSING COLUMNS PER ROW:")
print("=" * 50)
print(missing_counts.value_counts().sort_index())


In [ ]:
# Count missing values per row in the 49 columns
missing_counts = df[cols_49].isnull().sum(axis=1)

# Drop rows missing 20+ columns
rows_before = len(df)
df = df[missing_counts < 20].copy()
rows_after = len(df)

print(f"Dropped: {rows_before - rows_after:,} rows "
      f"({(rows_before - rows_after)/rows_before:.1%})")
print(f"Remaining: {rows_after:,} rows")

# Drop majorly empty columns
empty_cols = [col for col in df.columns if df[col].isnull().mean() > 0.4]
df.drop(columns=empty_cols, inplace=True, errors='ignore')

# Separate by data type
numeric_cols = df.select_dtypes(
    include=['float64', 'int64','float32', 'int32']
).columns.tolist()

categorical_cols = df.select_dtypes(
    include=['object','category']
).columns.tolist()

# Filling remaining small gaps with median
for col in numeric_cols:
    if df[col].isnull().sum() > 0 :
        df[col] = df[col].fillna(df[col].median())
# filling NaN in the emp_length column
df['emp_length'] = df['emp_length'].fillna('Unknown')

print(f"Missing values: {df.isnull().sum()}")
print(f"Default rate: {df['default'].mean():.2%}")

### Missing Data: Block Pattern Analysis

Investigation revealed that 29 credit detail columns
shared a correlated missing pattern. Row-level analysis
showed a bimodal distribution:

- **1,299,290 rows (95%)**: 0-7 columns missing
  → imputed with median (minimal impact)
- **67,527 rows (5%)**: 24-29 columns missing
  → dropped (imputing 24+ features would introduce
  noise rather than signal)

No rows fell in the 8-23 range, indicating a clean
structural split between complete and incomplete
credit records. This pattern likely reflects
differences in credit bureau reporting standards
across time periods or borrower segments.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

key_features = [
    'annual_inc', 'loan_amnt', 'int_rate',
    'dti', 'revol_bal', 'revol_util',
    'tot_cur_bal', 'fico_range_low', 'installment'
]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]

    # Plot histogram
    data = df[col].dropna()

    # Cap at 99th percentile for visualization
    # (so outliers don't squish the chart)
    cap = data.quantile(0.99)
    data_capped = data[data <= cap]

    ax.hist(data_capped, bins=50, color='steelblue',
            edgecolor='white', alpha=0.7)

    # Add median line
    median = data.median()
    ax.axvline(median, color='red', linestyle='--',
               label=f'Median: {median:,.0f}')

    # Add mean line
    mean = data.mean()
    ax.axvline(mean, color='orange', linestyle='--',
               label=f'Mean: {mean:,.0f}')

    skew = data.skew()
    ax.set_title(f'{col}\nskew={skew:.2f}', fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle(
    'Feature Distributions (capped at 99th percentile)\n'
    'Red=Median, Orange=Mean. Gap between them = skew',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('distributions.png', dpi=150,
            bbox_inches='tight')
plt.show()

# Feature Engineering

In [ ]:
# Flag missing employment data
df['emp_length_missing'] = df['emp_length'].isnull().astype(int)
# How much of their income goes to this loan payment?
df['installment_to_income'] = (
    df['installment'] / (df['annual_inc'] / 12 + 1)
)

# How big is this loan relative to their income?
df['loan_to_income'] = (
    df['loan_amnt'] / (df['annual_inc'] + 1)
)

# How much of their total credit are they using?
df['total_utilization'] = (
    df['tot_cur_bal'] / (df['tot_hi_cred_lim'] + 1)
)

# Revolving balance relative to income
df['revol_bal_to_income'] = (
    df['revol_bal'] / (df['annual_inc'] + 1)
)
# What percentage of their accounts are currently open?
df['open_to_total_ratio'] = (
    df['open_acc'] / (df['total_acc'] + 1)
)

# How many accounts opened recently vs total?
df['recent_activity_ratio'] = (
    df['acc_open_past_24mths'] / (df['total_acc'] + 1)
)
# High = opening many new accounts recently = seeking credit

# Average balance per account
df['avg_balance_per_acc'] = (
    df['tot_cur_bal'] / (df['open_acc'] + 1)
)

# Bankcard utilization relative to overall utilization
df['bc_vs_revol_util'] = (
        df['bc_util'] / (df['revol_util'] + 1)
    )
    # If this is high: credit card utilization is
    # higher than overall revolving utilization
    # Means credit cards are more maxed out
    # than other revolving accounts
# Total number of derogatory marks
df['total_derog_marks'] = (
    df['pub_rec'] +
    df['pub_rec_bankruptcies'] +
    df['tax_liens']
)
# Sum of bankruptcies + public records + tax liens
# 0 = clean history
# 3+ = serious credit problems

# Any current delinquency?
df['has_current_delinq'] = (
    (df['acc_now_delinq'] > 0).astype(int)
)
# Binary: are they behind on ANY payment right now?

# Total late accounts (any severity)
df['total_late_accounts'] = (
    df['num_tl_30dpd'] +
    df['num_tl_90g_dpd_24m'] +
    df['num_accts_ever_120_pd']
)

# Recent credit seeking intensity
df['credit_seeking_intensity'] = (
        df['inq_last_6mths'] *
        df['acc_open_past_24mths']
    )

# Average FICO score (from range)
df['fico_avg'] = (
    (df['fico_range_low'] + df['fico_range_high']) / 2
)
# The dataset gives a range like 700-704
# Average gives a single clean number: 702

# FICO range width
df['fico_range_width'] = (
    df['fico_range_high'] - df['fico_range_low']
)

# FICO relative to grade expectation
# Lower FICO than expected for the grade = riskier

print("Features created")

# Correlations of new features with target

In [ ]:
# Correlate all new features with default
new_features = [
    'installment_to_income', 'loan_to_income',
    'total_utilization', 'revol_bal_to_income',
    'open_to_total_ratio', 'recent_activity_ratio',
    'avg_balance_per_acc', 'total_derog_marks',
    'has_current_delinq', 'total_late_accounts',
    'credit_seeking_intensity', 'fico_avg',
    'fico_vs_grade_avg',
]

# Only check features that exist
new_features = [f for f in new_features if f in df.columns]

print("NEW FEATURE CORRELATIONS WITH DEFAULT:")
print("=" * 60)

correlations = {}
for col in new_features:
    corr = df[col].corr(df['default'])
    correlations[col] = corr

# Sort by absolute correlation
sorted_corr = sorted(
    correlations.items(),
    key=lambda x: abs(x[1]),
    reverse=True
)

for col, corr in sorted_corr:
    direction = "↑ riskier" if corr > 0 else "↓ safer"
    bar = "█" * int(abs(corr) * 200)
    print(f"  {col:30s} {corr:+.4f}  {direction:10s} {bar}")

# To prevent data leakage due to target encoding

In [ ]:
from sklearn.model_selection import train_test_split

df['term'] = (
    df['term'].astype(str)
    .str.strip()
    .str.replace(' months', '')
)
df['term'] = pd.to_numeric(df['term'], errors='coerce')
df['term'] = df['term'].fillna(df['term'].median())
X = df.drop(columns=['default'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
state_rates = X_train.copy()
state_rates['default'] = y_train
state_default = state_rates.groupby(
    'addr_state'
)['default'].mean()

X_train['state_default_rate'] = (
    X_train['addr_state'].map(state_default)
)
X_test['state_default_rate'] = (
    X_test['addr_state'].map(state_default)
)
# Test set uses rates calculated from TRAINING
# data only. No leakage.

# Same for fico_vs_grade_avg
fico_rates = X_train.copy()
fico_rates['fico_avg'] = (
    X_train['fico_range_low'] + X_train['fico_range_high']
) / 2
grade_avg = fico_rates.groupby(
    'sub_grade'
)['fico_avg'].mean()

# Apply training averages to both sets
X_train['fico_vs_grade_avg'] = (
    X_train['fico_avg'] - X_train['sub_grade'].map(grade_avg)
)
X_test['fico_vs_grade_avg'] = (
    X_test['fico_avg'] - X_test['sub_grade'].map(grade_avg))
X_train.drop(columns=['addr_state','fico_avg','grade', ], inplace=True, errors='ignore')
X_test.drop(columns=['addr_state','fico_avg','grade',], inplace=True, errors='ignore')

# Building model pipeline

In [ ]:
cols = [col for col in X_train.columns if X_train[col].dtype in ['object','category']]
print(df['term'].unique())
cols

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder, OneHotEncoder, StandardScaler
)
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# Define column groups
ordinal_cols = ['sub_grade', 'emp_length']
onehot_cols = ['home_ownership','verification_status','purpose','initial_list_status','application_type']
numeric_cols = X_train.select_dtypes(
    include=['float64', 'int64','float32', 'int32']
).columns.tolist()
# Build the preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal', OrdinalEncoder(
            categories=[
                sorted(X_train['sub_grade'].unique()), #subgrade
                ['Unknown','< 1 year','1 year','2 years','3 years','4 years','5 years','6 years','7 years','8 years','9 years','10+ years']
            ]
        ), ordinal_cols),

        ('onehot', OneHotEncoder(
            drop='first',
            handle_unknown='ignore'
        ), onehot_cols),

        ('numeric', StandardScaler(), numeric_cols),
    ]
)

# Combine preprocessing + model in one pipeline
ratio = (y_train==0).sum()/ (y_train==1).sum()
pipeline = Pipeline([
    ('preprocessor', preprocessor),
('model', XGBClassifier(scale_pos_weight=ratio,n_estimators=500, learning_rate=0.02, verbosity=1, reg_lambda=1, reg_alpha=0.4,eval_metric='auc'))
])
cols = ['int_rate', 'term', 'dti', 'state_default_rate', 'sub_grade', 'acc_open_past_24mths', 'recent_activity_ratio', 'fico_range_low', 'avg_cur_bal', 'loan_to_income', 'mths_since_recent_inq', 'mo_sin_old_rev_tl_op', 'home_ownership', 'num_rev_tl_bal_gt_0', 'emp_length', 'mort_acc', 'mths_since_recent_bc', 'total_bc_limit', 'loan_amnt']

pipeline.fit(X_train, y_train)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)
print(f"AUC score is {auc: .2f}")
print(f"Accuracy is: {acc : .2f}")


In [ ]:
import shap

explainer = shap.TreeExplainer(pipeline.named_steps['model'])
X_sample = X_test.sample(5000, random_state=42)
X_sample_transformed = pipeline.named_steps['preprocessor'].transform(X_sample)
shap_values = explainer.shap_values(X_sample_transformed)
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

shap.summary_plot(shap_values, X_sample_transformed, feature_names=feature_names)

In [ ]:
shap_importance = np.abs(shap_values).mean(axis=0)

important_features = [feature_names[i] for i in np.argsort(shap_importance)[::-1][:10]]
important_features = [
    f.split('__')[1] for f in important_features
]
print(important_features)

In [ ]:
top_indices = np.argsort(shap_importance)[::-1][:10]

important_features = {
    feature_names[i].split('__')[1]: shap_importance[i]
    for i in top_indices  # map each index to its clean name and score
}

print(important_features)

## Evaluation: Classification Report & Confusion Matrix

AUC measures how well the model *ranks* borrowers by risk globally,
but doesn't reflect what happens at any specific lending decision.

At the default threshold of 0.5:
- The model catches **69% of actual defaults** (recall)
- But flags **34% of good borrowers** as risky (false positive rate)

This tradeoff is expected: `scale_pos_weight` was set to penalize
missed defaults heavily, since a missed default costs the lender far
more than a rejected good borrower. The optimal threshold — explored
in the next section — will let us tune this tradeoff explicitly.

In [ ]:
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix
)
import matplotlib.pyplot as plt

# --- Classification Report ---
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Paid', 'Default']))

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Paid', 'Default']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')

# Annotate with rates
tn, fp, fn, tp = cm.ravel()
total = cm.sum()

ax.set_title(
    f"Confusion Matrix\n"
    f"True Negative Rate: {tn/(tn+fp):.1%}  |  "
    f"True Positive Rate (Recall): {tp/(tp+fn):.1%}",
    fontsize=10
)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOf {fn+tp:,} actual defaults → model caught {tp:,} ({tp/(tp+fn):.1%})")
print(f"Of {tn+fp:,} actual paid    → model flagged {fp:,} as risky ({fp/(tn+fp):.1%})")

## Precision-Recall Curve & KS Statistic

**Precision-Recall curve** is preferred over ROC on imbalanced data
because it focuses entirely on the minority class (defaults) and is
not flattered by the large number of true negatives.

**Average Precision (AP)** summarizes the curve as a single number —
analogous to AUC but for the PR space. Higher is better; random
baseline = the default rate (~22%).

**KS Statistic** (Kolmogorov-Smirnov) is the standard metric used
in banking and credit scoring to evaluate scorecard quality. It
measures the maximum separation between the True Positive Rate and
False Positive Rate curves across all thresholds.

| KS Range | Industry Interpretation |
|----------|------------------------|
| < 0.2    | Poor                   |
| 0.2–0.4  | Acceptable             |
| 0.4–0.6  | Good                   |
| > 0.6    | Very strong            |

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_curve
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── LEFT: Precision-Recall Curve ──────────────────────────────
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)
ap_score = average_precision_score(y_test, y_pred_proba)

ax = axes[0]
ax.plot(recalls, precisions, color='steelblue', lw=2)
ax.axhline(y=y_test.mean(), color='gray', linestyle='--',
           label=f'Random baseline ({y_test.mean():.2%})')

# Mark the current 0.5 threshold point
current_precision = precisions[np.searchsorted(thresholds_pr, 0.5)]
current_recall    = recalls[np.searchsorted(thresholds_pr, 0.5)]
ax.scatter(current_recall, current_precision,
           color='red', zorder=5, s=80, label=f'Threshold=0.5')

ax.set_xlabel('Recall (% of defaults caught)', fontsize=11)
ax.set_ylabel('Precision (% of flags that are real defaults)', fontsize=11)
ax.set_title(f'Precision-Recall Curve\nAverage Precision: {ap_score:.3f}', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)

# ── RIGHT: KS Statistic ───────────────────────────────────────
fpr, tpr, thresholds_roc = roc_curve(y_test, y_pred_proba)
ks_scores = tpr - fpr
ks_stat   = ks_scores.max()
ks_thresh = thresholds_roc[ks_scores.argmax()]

ax = axes[1]
ax.plot(thresholds_roc, tpr, color='steelblue', lw=2, label='True Positive Rate')
ax.plot(thresholds_roc, fpr, color='tomato',    lw=2, label='False Positive Rate')
ax.plot(thresholds_roc, ks_scores, color='green', lw=2,
        linestyle='--', label='KS = TPR - FPR')

ax.axvline(ks_thresh, color='green', linestyle=':', alpha=0.7,
           label=f'KS threshold: {ks_thresh:.3f}')
ax.annotate(f'KS = {ks_stat:.3f}',
            xy=(ks_thresh, ks_stat),
            xytext=(ks_thresh + 0.05, ks_stat - 0.08),
            fontsize=10, color='green',
            arrowprops=dict(arrowstyle='->', color='green'))

ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('Rate', fontsize=11)
ax.set_title(f'KS Statistic = {ks_stat:.3f}\n'
             f'Optimal threshold: {ks_thresh:.3f}', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Model Discrimination: PR Curve & KS Statistic',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pr_ks_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAverage Precision Score: {ap_score:.3f}")
print(f"KS Statistic:            {ks_stat:.3f}  (industry threshold: >0.2 = usable, >0.4 = good)")
print(f"KS-optimal threshold:    {ks_thresh:.3f}")

## Business Impact Summary

At the F1-optimal threshold of **0.530**, the model:
- **Catches 64% of actual defaults** — reducing expected loan losses
- **Approves 71% of creditworthy borrowers** — preserving revenue
- Outperforms a random classifier (22% precision baseline) by
  **~15 percentage points** in precision

A stricter threshold can increase default detection at the cost of
approving fewer good borrowers — this tradeoff is explicitly
controllable and should reflect the lender's cost ratio of
(cost of default) vs (cost of missed revenue).

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# ── Compute metrics across all thresholds ─────────────────────
thresholds = np.arange(0.1, 0.9, 0.005)

precisions_t, recalls_t, f1s_t = [], [], []
fpr_t = []

for t in thresholds:
    preds = (y_pred_proba >= t).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    fp = ((preds == 1) & (y_test == 0)).sum()
    tn = ((preds == 0) & (y_test == 0)).sum()

    precisions_t.append(p)
    recalls_t.append(r)
    f1s_t.append(f)
    fpr_t.append(fp / (fp + tn))

precisions_t = np.array(precisions_t)
recalls_t    = np.array(recalls_t)
f1s_t        = np.array(f1s_t)
fpr_t        = np.array(fpr_t)

# ── Find key thresholds ───────────────────────────────────────
f1_optimal_thresh    = thresholds[np.argmax(f1s_t)]
ks_optimal_thresh    = 0.511  # from KS analysis above

# Business threshold: catch at least 60% of defaults
# while keeping false positive rate below 20%
business_mask = (recalls_t >= 0.60) & (fpr_t <= 0.20)
if business_mask.any():
    business_thresh = thresholds[business_mask][
        np.argmax(f1s_t[business_mask])
    ]
else:
    business_thresh = f1_optimal_thresh

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# LEFT: metrics vs threshold
ax = axes[0]
ax.plot(thresholds, precisions_t, label='Precision', color='steelblue', lw=2)
ax.plot(thresholds, recalls_t,    label='Recall',    color='tomato',    lw=2)
ax.plot(thresholds, f1s_t,        label='F1 Score',  color='green',     lw=2)
ax.plot(thresholds, fpr_t,        label='False Positive Rate',
        color='orange', lw=2, linestyle='--')

for thresh, label, color in [
    (0.5,                 'Current (0.5)',  'gray'),
    (f1_optimal_thresh,   'F1-optimal',     'green'),
    (business_thresh,     'Business',       'purple'),
]:
    ax.axvline(thresh, linestyle=':', color=color, alpha=0.8, label=f'{label}: {thresh:.3f}')

ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Metrics vs Decision Threshold', fontsize=12)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# RIGHT: business impact at each threshold
approved_rate  = 1 - recalls_t  # % of defaults that slip through (missed)
rejection_rate = fpr_t          # % of good borrowers wrongly rejected

ax = axes[1]
ax.plot(thresholds, (1 - fpr_t) * 100,
        label='Good borrowers approved (%)', color='steelblue', lw=2)
ax.plot(thresholds, recalls_t * 100,
        label='Defaults caught (%)', color='tomato', lw=2)

for thresh, label, color in [
    (0.5,               'Current (0.5)', 'gray'),
    (f1_optimal_thresh, 'F1-optimal',    'green'),
    (business_thresh,   'Business',      'purple'),
]:
    ax.axvline(thresh, linestyle=':', color=color, alpha=0.8, label=f'{label}: {thresh:.3f}')

ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_title('Business Impact vs Threshold\n'
             '(Lender perspective)', fontsize=12)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Threshold Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────
print(f"\n{'THRESHOLD COMPARISON':^65}")
print("=" * 65)
print(f"{'Threshold':<18} {'Recall':>8} {'Precision':>10} "
      f"{'F1':>7} {'FPR':>8} {'Good Approved':>14}")
print(f"\nRecommended threshold: {f1_optimal_thresh:.3f}")
print("-" * 65)

for thresh, label in [
    (0.5,               'Current (0.5)'),
    (f1_optimal_thresh, 'F1-optimal'),
    (business_thresh,   'Business'),
    (ks_optimal_thresh, 'KS-optimal'),
]:
    idx = np.argmin(np.abs(thresholds - thresh))
    print(f"{label:<18} {recalls_t[idx]:>8.1%} {precisions_t[idx]:>10.1%} "
          f"{f1s_t[idx]:>7.3f} {fpr_t[idx]:>8.1%} "
          f"{(1 - fpr_t[idx]):>13.1%}")

## Model Calibration

A well-calibrated model produces probabilities that reflect reality —
a predicted probability of 0.3 should mean the borrower defaults
roughly 30% of the time. This matters in lending because the raw
probability score is often used directly to set interest rates or
credit limits, not just as a binary flag.

**The right plot** shows the predicted probability distributions for
defaulters and non-defaulters. Good separation between the two
distributions indicates strong discrimination — but does not
guarantee calibration.

**The left plot** (reliability diagram) checks calibration directly:
points above the diagonal mean the model *underestimates* default
risk; points below mean it *overestimates*. A model inflated by
`scale_pos_weight` will typically sit below the diagonal —
systematically overestimating default probability.

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── LEFT: Calibration Curve ───────────────────────────────────
ax = axes[0]

fraction_of_positives, mean_predicted_proba = calibration_curve(
    y_test, y_pred_proba, n_bins=10
)

ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly calibrated')
ax.plot(mean_predicted_proba, fraction_of_positives,
        marker='o', color='steelblue', lw=2, label='XGBoost')

ax.axhline(y=y_test.mean(), color='tomato', linestyle=':',
           label=f'True default rate ({y_test.mean():.2%})')

ax.set_xlabel('Mean Predicted Probability', fontsize=11)
ax.set_ylabel('Fraction of Actual Defaults', fontsize=11)
ax.set_title('Calibration Curve\n(Reliability Diagram)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── RIGHT: Probability Distribution ──────────────────────────
ax = axes[1]

ax.hist(y_pred_proba[y_test == 0], bins=50, alpha=0.6,
        color='steelblue', label='Paid', density=True)
ax.hist(y_pred_proba[y_test == 1], bins=50, alpha=0.6,
        color='tomato', label='Default', density=True)

ax.axvline(0.530, color='green', linestyle='--',
           label='Optimal threshold (0.530)')
ax.axvline(0.5, color='gray', linestyle=':',
           label='Default threshold (0.5)')

ax.set_xlabel('Predicted Probability', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Predicted Probability Distribution\nby True Class', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Model Calibration Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('calibration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean predicted probability:  {y_pred_proba.mean():.3f}")
print(f"True default rate:           {y_test.mean():.3f}")
print(f"Calibration gap:             {y_pred_proba.mean() - y_test.mean():.3f} "
      f"(positive = overestimates default risk)")

**Findings**: The model systematically overestimates default probability
(calibration curve below diagonal) — an expected consequence of
`scale_pos_weight`. The probability distributions of defaulters and
non-defaulters overlap significantly in the 0.3–0.7 range, reflecting
the inherent difficulty of credit risk prediction from origination-time
features alone. Isotonic regression or Platt scaling could improve
calibration if raw probability scores are used downstream.

## SHAP: Single-Prediction Explanation

The summary plot shows which features matter globally across all
predictions. But in a real lending decision, the question is:
**why was *this specific borrower* flagged?**

Two predictions are explained below using SHAP waterfall plots:
- **A high-confidence default**: a borrower the model was most
  certain would default
- **A high-confidence paid**: a borrower the model was most
  certain would repay

Each bar shows how much a feature *pushed the prediction higher
(red)* or *lower (blue)* relative to the model's average prediction.
This is the kind of explanation a lender would need to justify a
rejection decision — increasingly a regulatory requirement under
fair lending laws.

In [ ]:
import shap

# Pick one predicted default and one predicted paid
# to show the model's reasoning in both directions
default_indices = np.where((y_pred_proba >= 0.530) & (y_test.values == 1))[0]
paid_indices    = np.where((y_pred_proba < 0.530)  & (y_test.values == 0))[0]

# Take the most confidently predicted of each
most_confident_default = default_indices[np.argmax(y_pred_proba[default_indices])]
most_confident_paid    = paid_indices[np.argmin(y_pred_proba[paid_indices])]

# Get transformed feature names
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

# Transform the sample
X_sample = X_test.iloc[[most_confident_default, most_confident_paid]]
X_sample_transformed = pipeline.named_steps['preprocessor'].transform(X_sample)

# Compute SHAP values
explainer   = shap.TreeExplainer(pipeline.named_steps['model'])
shap_values = explainer.shap_values(X_sample_transformed)

# ── Plot 1: High-confidence default ──────────────────────────
print("=" * 60)
print("EXPLANATION 1: High-confidence predicted DEFAULT")
print("=" * 60)
shap.waterfall_plot(
    shap.Explanation(
        values         = shap_values[0],
        base_values    = explainer.expected_value,
        data           = X_sample_transformed[0],
        feature_names  = feature_names
    )
)

# ── Plot 2: High-confidence paid ─────────────────────────────
print("\n" + "=" * 60)
print("EXPLANATION 2: High-confidence predicted PAID")
print("=" * 60)
shap.waterfall_plot(
    shap.Explanation(
        values         = shap_values[1],
        base_values    = explainer.expected_value,
        data           = X_sample_transformed[1],
        feature_names  = feature_names
    )
)

### Key Findings

**Predicted default**: all features unanimously push toward default —
sub_grade, loan vintage, employment length, and debt-to-income ratios
all compound the risk signal with no mitigating factors.

**Predicted repayment**: driven by strong credit grade, high FICO,
and low debt burden relative to income.

Notably, engineered features (`loan_to_income`, `installment_to_income`,
`revol_bal_to_income`) appear as meaningful contributors in both
explanations — confirming they captured genuine signal beyond what
raw features provide.

The sub_grade feature dominates both predictions, which is expected —
LendingClub's own internal risk assessment is the single strongest
predictor of outcome. The model's value lies in combining this with
behavioral and financial ratio signals that sub_grade alone doesn't
capture.

In [ ]:
# ── Retrain on 10 features for deployment ────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import precision_recall_curve
import joblib

# ── Feature selection ─────────────────────────────────────────
features_10 = [
    'sub_grade', 'issue_year', 'term', 'dti',
    'loan_amnt', 'annual_inc', 'addr_state',
    'home_ownership', 'fico_range_low', 'emp_length', 'mort_acc'
]

X_deploy = df[features_10].copy()
y_deploy = df['default'].copy()

# ── Train/test split ──────────────────────────────────────────
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_deploy, y_deploy,
    test_size=0.2, stratify=y_deploy, random_state=42
)

# ── Engineer features (training data only) ────────────────────
# loan_to_income
X_train_d['loan_to_income'] = (
    X_train_d['loan_amnt'] / (X_train_d['annual_inc'] + 1)
)
X_test_d['loan_to_income'] = (
    X_test_d['loan_amnt'] / (X_test_d['annual_inc'] + 1)
)

# state_default_rate — computed from training only
state_rates = X_train_d.copy()
state_rates['default'] = y_train_d.values
state_default = state_rates.groupby('addr_state')['default'].mean()
global_default_rate = y_train_d.mean()

X_train_d['state_default_rate'] = (
    X_train_d['addr_state'].map(state_default)
)
X_test_d['state_default_rate'] = (
    X_test_d['addr_state'].map(state_default)
    .fillna(global_default_rate)
)

# Drop raw columns that have been replaced
X_train_d.drop(columns=['loan_amnt', 'annual_inc', 'addr_state'], inplace=True)
X_test_d.drop(columns=['loan_amnt', 'annual_inc', 'addr_state'], inplace=True)

# ── Define column groups ──────────────────────────────────────
ordinal_cols_d  = ['sub_grade', 'emp_length']
onehot_cols_d   = ['home_ownership']
numeric_cols_d  = ['issue_year', 'term', 'dti', 'fico_range_low',
                   'mort_acc', 'loan_to_income', 'state_default_rate']

# ── Build pipeline ────────────────────────────────────────────
preprocessor_d = ColumnTransformer(transformers=[
    ('ordinal', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
        categories=[
            sorted(X_train_d['sub_grade'].unique()),
            ['Unknown', '< 1 year', '1 year', '2 years', '3 years',
             '4 years', '5 years', '6 years', '7 years',
             '8 years', '9 years', '10+ years']
        ]
    ), ordinal_cols_d),
    ('onehot', OneHotEncoder(
        drop='first', handle_unknown='ignore'
    ), onehot_cols_d),
    ('numeric', StandardScaler(), numeric_cols_d),
])

ratio_d = (y_train_d == 0).sum() / (y_train_d == 1).sum()

pipeline_deploy = Pipeline([
    ('preprocessor', preprocessor_d),
    ('model', XGBClassifier(
        scale_pos_weight=ratio_d,
        n_estimators=500,
        learning_rate=0.02,
        max_depth=6,
        reg_lambda=1,
        reg_alpha=7,
        eval_metric='auc',
        random_state=42
    ))
])

# ── Train ─────────────────────────────────────────────────────
pipeline_deploy.fit(X_train_d, y_train_d)

# ── Evaluate ──────────────────────────────────────────────────
y_proba_d = pipeline_deploy.predict_proba(X_test_d)[:, 1]

auc_d = roc_auc_score(y_test_d, y_proba_d)
ap_d  = average_precision_score(y_test_d, y_proba_d)

print(f"10-feature deployment model:")
print(f"  ROC-AUC:           {auc_d:.3f}  (full model: 0.75)")
print(f"  Average Precision: {ap_d:.3f}  (full model: 0.439)")
print(f"  Features used:     {X_train_d.shape[1]}")

In [ ]:
# Save deployment model + metadata
joblib.dump({
    'pipeline': pipeline_deploy,
    'state_default_rates': state_default.to_dict(),
    'global_default_rate': global_default_rate,
    'feature_names': list(X_train_d.columns),
    'threshold': 0.530,
    'auc': auc_d,
}, 'credit_model.pkl')

print(f"Saved. Size: {os.path.getsize('credit_model.pkl') / 1024**2:.1f} MB")

In [ ]:
from google.colab import files
files.download('credit_model.pkl')